In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/ai4i2020.csv")

In [2]:
X = df[
    [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]
]

y = df["Machine failure"]

X.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min]
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


In [3]:
from sklearn.model_selection import train_test_split    

In [4]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y   #when splitting the data preserve the distribution of target variable y in both training and test sets
)

## Data Preprocessing

Before training the model, we need to prepare the features for Logistic Regression.

- `Type` is a categorical feature, so we use one-hot encoding to convert it into numerical values.
- The remaining features are numerical but have different scales, so we use standardization.
- The preprocessing is fitted only on the training data to avoid data leakage.
- The same transformations are then applied to the test data.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [6]:
categorical_cols = ["Type"]

numerical_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

In [7]:
preprocessor=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_cols),
        ("num",StandardScaler(),numerical_cols)
    ]
)

## Logistic Regression

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [9]:
lr_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [10]:
lr_cv_results = cross_validate(
    lr_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [11]:
for metric in scoring:
    val_scores = lr_cv_results[f"test_{metric}"]
    train_scores = lr_cv_results[f"train_{metric}"]

    print(
        f"{metric.upper():10s} "
        f"CV: {val_scores.mean():.3f} +/- {val_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION  CV: 0.761 +/- 0.109 | Train: 0.747
RECALL     CV: 0.199 +/- 0.028 | Train: 0.198
F1         CV: 0.312 +/- 0.026 | Train: 0.313
ROC_AUC    CV: 0.894 +/- 0.017 | Train: 0.897
PR_AUC     CV: 0.469 +/- 0.032 | Train: 0.469


## Logistic Regression — Baseline Model

Logistic Regression is used as the first baseline model to predict machine failures. The model is trained using the selected sensor features and product type, with preprocessing performed inside a pipeline.

We evaluate the model using **5-fold stratified cross-validation** on the training set. Precision, recall, F1-score, ROC-AUC, and PR-AUC (Average Precision) are used to assess its performance because machine failures are relatively rare.

### Results and Observations

- **Precision:** 0.761 — most predicted failures are actual failures.
- **Recall:** 0.199 — the model detects only about 20% of actual failures at the default classification threshold of 0.5.
- **F1-score:** 0.312 — the balance between precision and recall is limited by the low recall.
- **ROC-AUC:** 0.894 — the model shows good ability to distinguish between failure and non-failure cases.
- **PR-AUC:** 0.469 — provides a useful measure of performance for this imbalanced classification problem.

The training and cross-validation scores are close, suggesting no substantial overfitting. However, the low recall indicates that the default threshold misses many actual failures. This baseline will be compared with other classification models before deciding whether further tuning is needed.

## Decision Tree Classifier

In [12]:
from sklearn.tree import DecisionTreeClassifier

In [13]:
dt_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

In [14]:
dt_cv_results = cross_validate(
    dt_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [15]:
for metric in scoring:
    val_scores = dt_cv_results[f"test_{metric}"]
    train_scores = dt_cv_results[f"train_{metric}"]

    print(
        f"{metric.upper():10s} "
        f"CV: {val_scores.mean():.3f} +/- {val_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION  CV: 0.721 +/- 0.045 | Train: 1.000
RECALL     CV: 0.679 +/- 0.060 | Train: 1.000
F1         CV: 0.699 +/- 0.051 | Train: 1.000
ROC_AUC    CV: 0.835 +/- 0.030 | Train: 1.000
PR_AUC     CV: 0.503 +/- 0.071 | Train: 1.000


## Decision Tree — Baseline Model

A Decision Tree Classifier is used as the second baseline model to predict machine failures. The model uses the same selected features, preprocessing pipeline, and 5-fold stratified cross-validation strategy as Logistic Regression.

### Results and Observations

- **Precision:** 0.721 — about 72.1% of predicted failures are actual failures.
- **Recall:** 0.679 — the model detects approximately 67.9% of actual failures.
- **F1-score:** 0.699 — a better balance between precision and recall than the Logistic Regression baseline.
- **ROC-AUC:** 0.835 — the model shows good discrimination between failure and non-failure cases.
- **PR-AUC:** 0.503 — higher than the Logistic Regression baseline.

The training scores are 1.000 across all five metrics, while the cross-validation scores are lower. This indicates **significant overfitting**, suggesting that the tree has learned patterns specific to the training data that do not generalize equally well to unseen data.

Despite this overfitting, the Decision Tree achieves substantially higher recall and F1-score than Logistic Regression at the default classification threshold. We will consider controlling the tree's complexity during the hyperparameter-tuning stage.

## Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier

In [17]:
rf_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])

In [18]:
rf_cv_results = cross_validate(
    rf_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

In [19]:
for metric in scoring:
    val_scores = rf_cv_results[f"test_{metric}"]
    train_scores = rf_cv_results[f"train_{metric}"]

    print(
        f"{metric.upper():10s} "
        f"CV: {val_scores.mean():.3f} +/- {val_scores.std():.3f} | "
        f"Train: {train_scores.mean():.3f}"
    )

PRECISION  CV: 0.901 +/- 0.053 | Train: 1.000
RECALL     CV: 0.483 +/- 0.036 | Train: 1.000
F1         CV: 0.628 +/- 0.032 | Train: 1.000
ROC_AUC    CV: 0.960 +/- 0.011 | Train: 1.000
PR_AUC     CV: 0.771 +/- 0.026 | Train: 1.000


## Random Forest — Baseline Model

Random Forest is used as the third baseline model to predict machine failures. It combines multiple Decision Trees, each trained on a bootstrap sample of the training data. Random feature selection at each split helps introduce diversity among the trees, and their predictions are aggregated to produce the final prediction.

The model is evaluated using the same 5-fold stratified cross-validation strategy and scoring metrics as the previous models.

### Results and Observations

- **Precision:** 0.901 — approximately 90.1% of predicted failures are actual failures.
- **Recall:** 0.483 — the model detects approximately 48.3% of actual failures.
- **F1-score:** 0.628 — a better balance between precision and recall than Logistic Regression, but lower than the Decision Tree baseline.
- **ROC-AUC:** 0.960 — indicates excellent discrimination between failure and non-failure cases.
- **PR-AUC:** 0.771 — the highest PR-AUC among the three models evaluated so far.

The training scores are 1.000 across all five metrics, while the cross-validation scores are lower, indicating substantial overfitting. However, the model shows promising performance, particularly in precision, ROC-AUC, and PR-AUC.

Random Forest will be considered for hyperparameter tuning to investigate whether its generalization performance can be improved.